# Benchmark: Strands decider (2B) on our routing tests

**What.** A zero-shot comparison. [`StrandsAgents/strands-decider-2B-hobson-v19`](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19) is a ready-made **decision model**: Qwen3.5-2B-Base with a LoRA adapter and a small readout head. Given a *state* (text) and a *choice* question with named options, it returns a calibrated probability per option. Routing is exactly such a choice: the state is the conversation, and the options are the agents with their descriptions.

**The tests**, the same files as `modernbert.ipynb`, none of which this model was trained on:

| ID | Test | Size |
|---|---|---|
| E11 | First message only: pick 1 of 5 agents | 100 messages, 10 companies |
| E10 | Realistic chat history: route the latest user turn | 35 conversations, 193 turns, 13 companies, 3 agents each |
| E3 | Banking chat history (benchmark + extended) | 225 conversations, 5 agents |

**Compared with** our best router so far (run 9: Nemotron-1B encoder + trained head) from the experiment ledger.

Nothing here is trained: the decider is used as published.

## 1 — Setup

Install the decider package, pin transformers to the same version as `modernbert.ipynb`, and show the GPU.

In [ ]:
!pip install -q "strands-decider" "transformers==5.17.0"
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
      "| bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## 2 — Test data

The test files live in the repo on the experiment branch. They are cloned (or refreshed) from that branch.

In [ ]:
import json, subprocess, time
from collections import Counter
from pathlib import Path

DATA_BRANCH = "exp/benchmark-strands-decider"
repo = Path("llms-the-hard-way")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                    "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
DATA_DIR = repo / "data" / "modernbert_router"

ft = json.load(open(DATA_DIR / "first_turn" / "messages.json"))
real_convs = json.load(open(DATA_DIR / "realistic" / "conversations.json"))
banking_all = json.load(open(DATA_DIR / "banking" / "benchmark.json")) + json.load(open(DATA_DIR / "banking" / "extended.json"))
BANKING_AGENTS = {                                          # exactly as in modernbert.ipynb, Cell 37
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
print(f"E11 first turn: {len(ft['messages'])} messages | E10 realistic: {len(real_convs)} conversations, "
      f"{sum(len(c['user_turns']) for c in real_convs)} turns | banking: {len(banking_all)} conversations")

## 3 — Load the decider

`load_engine` downloads the base model (Qwen3.5-2B-Base) and the checkpoint's adapter and head. The checkpoint asks for **bfloat16**. A T4 has no bfloat16 support, so in that case the model is moved to **float32** (about 8 GB for 2B parameters). A first question from the model card checks that it answers sensibly.

In [ ]:
from strands_decider.infer import load_engine
from strands_decider.schema import ChoiceQuestion

CHECKPOINT = "StrandsAgents/strands-decider-2B-hobson-v19"
t0 = time.perf_counter()
engine = load_engine(CHECKPOINT, device="cuda")
if not torch.cuda.is_bf16_supported():                       # e.g. a T4: run in float32 instead of bfloat16
    engine.model.to(torch.float32)
print(f"loaded {CHECKPOINT} in {time.perf_counter() - t0:.0f}s | dtype {next(engine.model.torso.parameters()).dtype}")

demo = engine.ask("Help! My payouts have been failing for 3 days!",
                  {"team": ChoiceQuestion(instructions="Which team should handle this?",
                                          criteria={"billing": None, "sales": None, "retail": None})})
print("model card example ->", demo.answers["team"].choice, demo.answers["team"].probabilities)

## 4 — The routing question

- **State:** the conversation as a plain transcript, one line per message, with the latest user message repeated at the end so the question is unambiguous.
- **Question:** *"Which agent should handle the user's latest message?"*
- **Options:** each agent name with its description.

`window` keeps only the last N user turns (with the replies between them), to compare full history with a recent window.

In [ ]:
INSTRUCTIONS = "Which agent should handle the user's latest message?"


def state_text(messages, window=None):
    """Transcript of the conversation up to the latest user message, optionally only the last `window` user turns."""
    if window is not None:
        user_idx = [i for i, m in enumerate(messages) if m["role"] == "user"]
        messages = messages[user_idx[-window]:] if len(user_idx) > window else messages
    lines = [f"{m['role']}: {m['content']}" for m in messages]
    return "Conversation:\n" + "\n".join(lines) + f"\n\nLatest user message: {messages[-1]['content']}"


def decide(messages, agents, window=None):
    """-> (chosen agent, probabilities {agent: p}) for the latest user message."""
    q = ChoiceQuestion(instructions=INSTRUCTIONS, criteria=dict(agents))
    ans = engine.ask(state_text(messages, window), {"route": q}).answers["route"]
    probs = dict(sorted(ans.probabilities.items(), key=lambda kv: -kv[1]))
    return ans.choice, probs


print(state_text(real_convs[0]["messages"][:5]))
print(decide(real_convs[0]["messages"][:5], real_convs[0]["agents"]))

## 5 — E11: first message only (100 messages, 5 agents each)

Top-1 and top-2 per message type, and every mistake. Our router (run 9) scored **93 / 100 top-1, 97 / 100 top-2**.

In [ ]:
t0 = time.perf_counter()
e11 = []
for m in ft["messages"]:
    agents = ft["companies"][m["company"]]
    pick, probs = decide([{"role": "user", "content": m["message"]}], agents)
    e11.append({**m, "pick": pick, "top2": list(probs)[:2], "p": probs[pick]})
print(f"E11: {len(e11)} decisions in {time.perf_counter() - t0:.0f}s\n")

print(f"{'type':10s} | {'msgs':>4} | {'top-1':>9} | {'top-2':>9}")
for tt in ["clear", "short", "indirect", "close", "noisy", "ALL"]:
    rows = [r for r in e11 if tt == "ALL" or r["type"] == tt]
    print(f"{tt:10s} | {len(rows):>4} | {sum(r['pick'] == r['target_agent'] for r in rows):>4} / {len(rows):<3} | "
          f"{sum(r['target_agent'] in r['top2'] for r in rows):>4} / {len(rows):<3}")
print("\nmistakes:")
for r in [r for r in e11 if r["pick"] != r["target_agent"]]:
    print(f"  {r['type']:8s} | {r['message'][:60]:60s} | expected {r['target_agent'].replace(' Agent', ''):26s} | got {r['pick'].replace(' Agent', '')} ({r['p']:.2f})")

## 6 — E10: realistic chat history (193 turns)

Every user turn is routed with the conversation so far, assistant replies included, in two variants:
- **full** history;
- a **window** of the last 3 user turns.

The decider has no "current agent" input, so each turn is routed independently. Our router (run 9) scored **154 / 193**, and its cross-only head **158 / 193**.

In [ ]:
TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]
t0 = time.perf_counter()
e10 = []
for conv in real_convs:
    for u in conv["user_turns"]:
        prefix = conv["messages"][:u["message_index"] + 1]
        full, _ = decide(prefix, conv["agents"])
        win, _ = decide(prefix, conv["agents"], window=3)
        e10.append({"id": conv["id"], "type": u["turn_type"], "expected": u["expected_agent"],
                    "message": prefix[-1]["content"], "full": full, "window 3": win})
print(f"E10: {len(e10)} turns x 2 variants in {time.perf_counter() - t0:.0f}s\n")

print(f"{'turn type':14s} | {'turns':>5} | {'full history':>12} | {'window 3':>10}")
for tt in TURN_TYPES + ["ALL"]:
    rows = [r for r in e10 if tt == "ALL" or r["type"] == tt]
    if rows:
        print(f"{tt:14s} | {len(rows):>5} | {sum(r['full'] == r['expected'] for r in rows):>6} / {len(rows):<4} | "
              f"{sum(r['window 3'] == r['expected'] for r in rows):>5} / {len(rows):<4}")
print("\nfull-history mistakes (first 15):")
for r in [r for r in e10 if r["full"] != r["expected"]][:15]:
    print(f"  {r['id']:9s} {r['type']:12s} | {r['message'][:50]:50s} | expected {r['expected'].replace(' Agent', ''):22s} | got {r['full'].replace(' Agent', '')}")

## 7 — Banking chat history (225 conversations, 5 agents)

The banking test from `modernbert.ipynb` (benchmark + extended), with the last user message routed. Our router (run 9) scored **193 / 225** (benchmark **43 / 50**).

In [ ]:
t0 = time.perf_counter()
bank = []
for ex in banking_all:
    pick, _ = decide(ex["messages"], BANKING_AGENTS)
    bank.append(pick == ex["target_agent"])
print(f"banking: {sum(bank[:50])} / 50 benchmark | {sum(bank)} / {len(bank)} all | {time.perf_counter() - t0:.0f}s")

## 8 — Summary next to our router

Our numbers are from the experiment ledger (run 9: Nemotron-1B encoder + trained head, `exp/phase3b-nemotron`).

In [ ]:
ours = {"E11 first turn, top-1 (100)": 93, "E11 first turn, top-2 (100)": 97,
        "E10 chat history (193)": 154, "banking benchmark (50)": 43, "all banking (225)": 193}
dec = {"E11 first turn, top-1 (100)": sum(r["pick"] == r["target_agent"] for r in e11),
       "E11 first turn, top-2 (100)": sum(r["target_agent"] in r["top2"] for r in e11),
       "E10 chat history (193)": max(sum(r["full"] == r["expected"] for r in e10), sum(r["window 3"] == r["expected"] for r in e10)),
       "banking benchmark (50)": sum(bank[:50]), "all banking (225)": sum(bank)}
print(f"{'test':32s} | {'our router (run 9)':>18} | {'strands decider':>15}")
for k in ours:
    print(f"{k:32s} | {ours[k]:>18} | {dec[k]:>15}")
print("\n(E10 decider = the better of full history / window 3)")